<a href="https://colab.research.google.com/github/c-marq/AI-Thinking-CAI1001C/blob/main/04-Data-Wrangling/Guided-Project/GP04-Customer-Data-Cleanup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Guided Project 4: Customer Data Cleanup
## Chapter 4: Data Wrangling and Preparation

**AI Thinking: A Hands-On Introduction to Artificial Intelligence**

---

### Scenario

You've just been hired as a data assistant at **TechFix Hialeah**, a small laptop and computer repair shop. The owner, Eddie, has been tracking repair orders in a spreadsheet, but the data is a mess. Before the shop can analyze trends—like which services are most popular or which technician gets the best ratings—the data needs to be cleaned up.

Your job: **Load, explore, and clean this dataset using pandas.**

---

### What You'll Practice
- Loading a CSV file into pandas
- Exploring data with `.shape`, `.info()`, `.describe()`, `.head()`
- Finding and handling missing values
- Removing duplicate rows
- Standardizing messy text
- Filtering out bad data
- Basic analysis on clean data

---
## Part 1: Setup and Load the Data

First, let's import pandas and load our dataset.

In [ ]:
# Import pandas
import pandas as pd

# Load the dataset
df = pd.read_csv('tech_repair_orders.csv')

# Quick look at the first few rows
df.head()

---
## Part 2: Explore the Data

Before we clean anything, let's understand what we're working with. Think of this like a mechanic inspecting a car before starting repairs — you need to know what's wrong first.

In [ ]:
# How big is our dataset?
print("Dataset shape:", df.shape)
print(f"That's {df.shape[0]} rows and {df.shape[1]} columns")

In [ ]:
# What columns do we have and what types are they?
df.info()

In [ ]:
# Statistical summary of numerical columns
df.describe()

### 🤔 What do you notice?

Look at the output from `.info()` and `.describe()`. Can you spot any problems?

- Do all columns have the same count of non-null values?
- Are the min/max values for price and rating reasonable?
- Anything else that looks off?

---
## Part 3: Find Missing Values

Let's get a clear picture of what's missing in our data.

In [ ]:
# Count missing values in each column
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# What percentage of each column is missing?
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
print("\nMissing value percentages:")
print(missing_pct)

---
## Part 4: Handle Missing Values

Now let's decide how to handle each missing column:

- **customer_name**: Fill with `'Unknown'` (we still want the order data)
- **price**: Fill with the **median** price (better than mean for skewed data)
- **technician**: Fill with `'Unknown'`
- **customer_rating**: Fill with the **median** rating (not everyone rates, this is common)

In [ ]:
# Fill missing customer names with 'Unknown'
df['customer_name'] = df['customer_name'].fillna('Unknown')

# Fill missing prices with the median price
median_price = df['price'].median()
print(f"Median price: ${median_price:.2f}")
df['price'] = df['price'].fillna(median_price)

# Fill missing technician names with 'Unknown'
df['technician'] = df['technician'].fillna('Unknown')

# Fill missing ratings with the median rating
median_rating = df['customer_rating'].median()
print(f"Median rating: {median_rating}")
df['customer_rating'] = df['customer_rating'].fillna(median_rating)

In [ ]:
# Verify: any missing values left?
print("Missing values after cleanup:")
print(df.isnull().sum())

💡 **Key Insight**: We used the **median** instead of the **mean** for price and rating. Why? The median isn't affected by extreme values (outliers). If one repair cost $500 and the rest were around $80, the mean would be pulled way up, but the median stays in the middle.

---
## Part 5: Remove Duplicates

Sometimes the same order gets recorded twice. Let's check.

In [ ]:
# How many duplicate rows do we have?
print(f"Duplicate rows found: {df.duplicated().sum()}")
print(f"Rows before removing duplicates: {len(df)}")

# Remove duplicates
df = df.drop_duplicates()

print(f"Rows after removing duplicates: {len(df)}")

---
## Part 6: Standardize Text

Let's look at how technician names and service types are recorded. You'd expect them to be consistent, but...

In [ ]:
# Check unique technician names
print("Technician names (before cleaning):")
print(df['technician'].unique())

In [ ]:
# Check unique service types
print("Service types (before cleaning):")
print(df['service_type'].unique())

See the problem? "Eddie", "eddie", "EDDIE", and " eddie " are all the same person, but pandas treats them as different values. Same issue with service types.

The fix: **convert everything to lowercase** and **remove extra spaces**.

In [ ]:
# Standardize technician names: lowercase + strip whitespace
df['technician'] = df['technician'].str.lower().str.strip()

# Standardize service types: lowercase + strip whitespace
df['service_type'] = df['service_type'].str.lower().str.strip()

# Check the results
print("Technician names (after cleaning):")
print(df['technician'].unique())

print("\nService types (after cleaning):")
print(df['service_type'].unique())

Much cleaner! Now "Eddie", "eddie", "EDDIE", and " eddie " are all just "eddie".

🔧 **Pro Tip**: Always standardize text columns early in your cleaning process. This is the #1 cause of "my analysis looks wrong" problems.

---
## Part 7: Filter Out Bad Data

Some values just don't make sense. Let's find and remove them.

In [ ]:
# Check for negative prices (shouldn't exist!)
bad_prices = df[df['price'] < 0]
print(f"Orders with negative prices: {len(bad_prices)}")
print(bad_prices[['order_id', 'service_type', 'price']])

In [ ]:
# Check for ratings outside the 1-5 scale
bad_ratings = df[(df['customer_rating'] < 1) | (df['customer_rating'] > 5)]
print(f"Orders with invalid ratings: {len(bad_ratings)}")
print(bad_ratings[['order_id', 'customer_name', 'customer_rating']])

In [ ]:
# Check for negative days to complete
bad_days = df[df['days_to_complete'] < 0]
print(f"Orders with negative days: {len(bad_days)}")
print(bad_days[['order_id', 'service_type', 'days_to_complete']])

In [ ]:
# Remove all bad data
print(f"Rows before filtering: {len(df)}")

# Keep only positive prices
df = df[df['price'] > 0]

# Keep only valid ratings (1-5)
df = df[(df['customer_rating'] >= 1) & (df['customer_rating'] <= 5)]

# Keep only positive days to complete
df = df[df['days_to_complete'] > 0]

print(f"Rows after filtering: {len(df)}")

⚠️ **Common Pitfall**: Notice how we used `&` (not `and`) and put each condition in parentheses when combining filters. This is a pandas-specific rule that trips up many beginners!

```python
# WRONG — will cause an error:
# df[df['rating'] >= 1 and df['rating'] <= 5]

# CORRECT:
df[(df['rating'] >= 1) & (df['rating'] <= 5)]
```

---
## Part 8: Verify Our Clean Data

Let's do a final check to make sure everything looks good.

In [ ]:
# Final data check
print("=== CLEAN DATA SUMMARY ===")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"\nPrice range: ${df['price'].min():.2f} - ${df['price'].max():.2f}")
print(f"Rating range: {df['customer_rating'].min()} - {df['customer_rating'].max()}")
print(f"Days range: {df['days_to_complete'].min()} - {df['days_to_complete'].max()}")

---
## Part 9: Quick Analysis on Clean Data

Now that our data is clean, let's answer some business questions for Eddie!

In [ ]:
# What's the average price per service?
print("Average price by service type:")
print(df.groupby('service_type')['price'].mean().round(2).sort_values(ascending=False))

In [ ]:
# Which technician has the best average rating?
print("Average rating by technician:")
print(df.groupby('technician')['customer_rating'].mean().round(2).sort_values(ascending=False))

In [ ]:
# What are the most common services?
print("Number of orders by service type:")
print(df['service_type'].value_counts())

In [ ]:
# Which device brand comes in most often?
print("Repairs by device brand:")
print(df['device_brand'].value_counts())

In [ ]:
# Quick bar chart: orders by service type
import matplotlib.pyplot as plt

df['service_type'].value_counts().plot(kind='barh')
plt.title('Number of Orders by Service Type')
plt.xlabel('Number of Orders')
plt.tight_layout()
plt.show()

In [ ]:
# Quick bar chart: average rating by technician
df.groupby('technician')['customer_rating'].mean().plot(kind='bar')
plt.title('Average Customer Rating by Technician')
plt.ylabel('Rating (1-5)')
plt.xlabel('Technician')
plt.ylim(0, 5)
plt.tight_layout()
plt.show()

---
## 🎓 What We Did

Let's recap the full data wrangling pipeline we just completed:

1. **Loaded** the data from a CSV file
2. **Explored** with `.shape`, `.info()`, `.describe()`, `.head()`
3. **Found missing values** with `.isnull().sum()`
4. **Filled missing values** with `.fillna()` using median and 'Unknown'
5. **Removed duplicates** with `.drop_duplicates()`
6. **Standardized text** with `.str.lower().str.strip()`
7. **Filtered bad data** by removing negative values and out-of-range ratings
8. **Verified** the clean dataset
9. **Analyzed** the clean data with `.groupby()` and `.value_counts()`

This is the same workflow you'd use on any messy dataset — in school, at work, or in your own projects.

---

**Great work!** 🎉 You just wrangled your first real dataset.